# 🌑 Apollo Landing Site Thermal Model

## 1-D Lunar Thermal Simulation — Clean & Modular Version

This notebook simulates how temperature changes with **depth and time** in the lunar soil
(regolith) beneath Apollo landing sites.  It is organised so that:

- **All user settings live in one place** (Section 1 — Configuration).
- **Functions are in separate `.py` files** (the `lunar/` package) — easy to read and reuse.
- **The notebook just calls those functions** — keeping each cell short and focused.

### What you can do here

| Section | What it does |
|---|---|
| 1 — Configuration | Set location, density model, and physical parameters |
| 2 — Load DEM | Load the LOLA elevation map |
| 3 — Run Model | Simulate temperature over several lunar days |
| 4 — Diurnal Cycles | Plot temperature vs time at different depths |
| 5 — Heatmap | 2-D view of temperature evolution |
| 6 — Apollo Comparison | Validate against real drill-hole measurements |
| 7 — Model Comparison | Compare Discrete vs Hayne density models |
| 8 — Sensitivity Analysis | How does one parameter shift the temperatures? |
| 9 — Single Point Analysis | Run two models at any list of locations |
| 10 — Batch Processing | Process many locations from a list |

| 1b | Parameter Panel | Adjust key values with sliders — no code editing needed |
| 2b | Terrain Map | Global DEM and zoomed local elevation around the target |
| 2c | Horizon Profile | Polar view of blocking terrain in every direction |
| 2d | Illumination Timeline | Sunlight, shadow, and absorbed flux over one lunar day |
| 2e | Temperature Map | Estimated surface temperature across the local terrain |
| 11 | Density Model Viewer | Interactive density-vs-depth profile with live updates |

### Background: what is being modelled?

The Moon has no atmosphere, so its surface swings from ~100 K (night) to ~390 K (day)
over each 29.5-Earth-day lunar cycle.  Beneath the surface this swing damps out quickly:
by 1 m depth the temperature is nearly constant year-round.  This notebook solves the
1-D heat-conduction equation through the regolith to predict that depth profile.

**Apollo 15 (1971) and Apollo 17 (1972) drilled ~2 m into the surface** and measured
temperatures directly.  We compare our model predictions against those measurements.

---
*Package structure: `lunar/constants.py`, `models.py`, `dem.py`, `horizon.py`,
`solar.py`, `solver.py`, `analysis.py`, `plots.py`*


## 0. Imports

Run this cell first — it loads all modules and sets the plot style.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Local package — all the physics lives here
from lunar import constants, models, dem, horizon, solver, analysis, plots

print("✓ All modules loaded")
print(f"  Physical constants  : lunar.constants")
print(f"  Density models      : lunar.models")
print(f"  DEM loading         : lunar.dem")
print(f"  Horizon computation : lunar.horizon")
print(f"  Solar geometry      : lunar.solar")
print(f"  Thermal solver      : lunar.solver")
print(f"  Post-processing     : lunar.analysis")
print(f"  Plotting            : lunar.plots")


## 1. Configuration

**Edit only this cell** to change the simulation.  Everything else runs automatically.

### Quick-start recipes

| Scenario | Settings |
|---|---|
| Apollo 15 validation | `LAT=26.1323`, `LON=3.6285`, `MODEL='discrete'` |
| Apollo 17 validation | `LAT=20.1911`, `LON=30.7723`, `MODEL='discrete'` |
| Hayne 2017 global   | `MODEL='hayne_exponential'`, `H_PARAM=0.07` |
| Cold bias fix        | Increase `SUNSCALE` (try 1.10–1.15) |
| Hot bias fix         | Decrease `SUNSCALE` (try 0.95–1.00) |

### Parameter guide

| Parameter | Typical range | Effect |
|---|---|---|
| `SUNSCALE` | 0.95 – 1.15 | Higher → more solar energy → hotter surface |
| `ALBEDO` | 0.07 – 0.14 | Higher → more reflection → cooler surface |
| `EMISSIVITY` | 0.90 – 0.98 | Higher → better IR radiator → cooler surface |
| `CHI` | 1.5 – 4.0 | Controls how conductivity rises with temperature |
| `H_PARAM` | 0.03 – 0.15 m | Hayne scale height; larger = slower compaction |
| `NDAYS` | 2 – 10 | More days = better spin-up (3–5 is usually enough) |


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║             SIMULATION CONFIGURATION — EDIT HERE                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Location ──────────────────────────────────────────────────────────────────
LAT = 26.1323    # Target latitude  (degrees, positive = north)
LON =  3.6285    # Target longitude (degrees, 0–360 east)
# Apollo 15: LAT=26.1323, LON=3.6285
# Apollo 17: LAT=20.1911, LON=30.7723

# ── Density model ─────────────────────────────────────────────────────────────
MODEL = 'discrete'       # 'discrete' | 'hayne_exponential' | 'custom'
# 'discrete'          → sharp layers from Apollo drill cores  (recommended for Apollo sites)
# 'hayne_exponential' → smooth exponential from Diviner data  (good for other locations)
# 'custom'            → edit lunar/models.py: density_custom() / k_solid_custom()

H_PARAM = 0.07    # Hayne scale height OR discrete Layer-1 thickness (metres)
                  # 0.07 m = 7 cm  (Hayne et al. 2017 global average)
                  # Adjust this when running sensitivity analysis on H.

# ── Energy balance ────────────────────────────────────────────────────────────
SUNSCALE   = 1.10   # Solar flux multiplier (1.0 = nominal; 1.10 = 10% more solar energy)
ALBEDO     = 0.09   # Fraction of sunlight reflected (higher → cooler surface)
EMISSIVITY = 0.95   # IR emissivity (higher → better heat radiator → cooler)
CHI        = 2.7    # Radiative conductivity exponent (controls k ∝ T³ term)

# ── Simulation ────────────────────────────────────────────────────────────────
NDAYS  = 3          # Lunar days to simulate (first N-1 = spin-up, last = analysis)
T_INIT = 250.0      # Initial uniform temperature (K)

# ── Derived values (do not edit) ─────────────────────────────────────────────
MODEL_ID = models.MODEL_ID_MAP[MODEL]
models.set_hayne_h(H_PARAM)
models.set_layer1_h(H_PARAM)

print(f"\n{'═'*60}")
print("CONFIGURATION SUMMARY")
print(f"{'═'*60}")
print(f"  Location  : {LAT:.4f}°N, {LON:.4f}°E")
print(f"  Model     : {MODEL}  (model_id={MODEL_ID})")
print(f"  H-param   : {H_PARAM*100:.1f} cm")
print(f"  SUNSCALE  : {SUNSCALE}")
print(f"  ALBEDO    : {ALBEDO}")
print(f"  EMISSIVITY: {EMISSIVITY}")
print(f"  CHI       : {CHI}")
print(f"  Sim days  : {NDAYS}  ({NDAYS * 29.53:.1f} Earth days)")
print(f"{'═'*60}")


## 1b. Interactive Parameter Panel

Use the sliders below to adjust key model parameters **without editing any code**.
When you are happy with the values, click **Apply & update globals** and then
re-run **Section 3** to see the effect on the simulated temperatures.

> **Tip:** The Density Model Viewer in Section 11 lets you preview the density
> profile instantly as you move the surface-density slider.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

_style  = {'description_width': '210px'}
_layout = widgets.Layout(width='520px')

w_rho = widgets.IntSlider(
    value=1100, min=800, max=1400, step=50,
    description='Surface density (kg/m3)',
    style=_style, layout=_layout,
)
w_sun = widgets.FloatSlider(
    value=SUNSCALE, min=0.90, max=1.20, step=0.01,
    description='Solar scale (SUNSCALE)',
    style=_style, layout=_layout,
)
w_albedo = widgets.FloatSlider(
    value=ALBEDO, min=0.06, max=0.16, step=0.005,
    description='Albedo',
    style=_style, layout=_layout,
)
w_chi = widgets.FloatSlider(
    value=CHI, min=1.5, max=5.0, step=0.1,
    description='Chi (radiative k)',
    style=_style, layout=_layout,
)
w_h = widgets.FloatSlider(
    value=H_PARAM, min=0.02, max=0.15, step=0.01,
    description='Layer-1 thickness (m)',
    style=_style, layout=_layout,
)
w_ndays = widgets.IntSlider(
    value=NDAYS, min=2, max=8, step=1,
    description='Simulation days',
    style=_style, layout=_layout,
)
apply_btn = widgets.Button(
    description='Apply & update globals',
    button_style='success',
    layout=widgets.Layout(width='240px', margin='10px 0 0 0'),
)
out_panel = widgets.Output()

help_html = (
    '<div style="background:#f0f8ff;border-radius:8px;padding:10px 14px;'
    'font-family:monospace;font-size:12px;margin-bottom:6px;">'
    '<b>What each parameter does:</b><br>'
    '<b>Surface density</b> — How loosely packed the top soil is '
    '(lower = fluffier, heats up faster). Apollo data: ~1100 kg/m3.<br>'
    '<b>Solar scale</b> — Multiplies the sunlight. '
    '1.0 = standard; try 1.05-1.15 if model runs cold.<br>'
    '<b>Albedo</b> — Fraction of sunlight reflected. '
    'Higher = cooler surface. Lunar soil: 0.07-0.12.<br>'
    '<b>Chi</b> — Controls radiative heat conduction at high temperatures.<br>'
    '<b>Layer-1 thickness</b> — Depth of fluffy top dust (Apollo: ~7 cm).<br>'
    '<b>Simulation days</b> — More days = better spin-up (3-5 is usually fine).'
    '</div>'
)

def on_apply(_):
    global SUNSCALE, ALBEDO, CHI, H_PARAM, NDAYS
    with out_panel:
        clear_output()
        SUNSCALE = w_sun.value
        ALBEDO   = w_albedo.value
        CHI      = w_chi.value
        H_PARAM  = w_h.value
        NDAYS    = w_ndays.value
        models.set_rho_surface(w_rho.value)
        models.set_hayne_h(H_PARAM)
        models.set_layer1_h(H_PARAM)
        print(f'Globals updated — re-run Section 3 to apply.')
        print(f'  Surface density : {w_rho.value} kg/m3')
        print(f'  SUNSCALE        : {SUNSCALE:.2f}')
        print(f'  ALBEDO          : {ALBEDO:.4f}')
        print(f'  CHI             : {CHI:.1f}')
        print(f'  Layer thickness : {H_PARAM*100:.0f} cm')
        print(f'  Simulation days : {NDAYS}')

apply_btn.on_click(on_apply)
display(widgets.HTML(help_html),
        w_rho, w_sun, w_albedo, w_chi, w_h, w_ndays,
        apply_btn, out_panel)

## 2. Load DEM and Extract Target Point

This cell loads the LOLA Digital Elevation Model (DEM) from the current directory
and snaps your target coordinates to the nearest pixel.

**Tip:** place `LDEM_*.LBL` and `LDEM_*.IMG` (or `.JP2`) next to this notebook.
Higher-resolution files are preferred automatically.


In [ ]:
# ── Load the elevation grid ────────────────────────────────────────────────────
ELEV_M, PIXEL_M, MAP_RES, ldem_label = dem.load_ldem()

# ── Extract the target point ──────────────────────────────────────────────────
(ROW, COL,
 ACTUAL_LAT, ACTUAL_LON,
 ELEVATION,
 SLOPE, ASPECT) = dem.extract_point(LAT, LON, ELEV_M, PIXEL_M, MAP_RES)

print(f"\n{'═'*60}")
print("TARGET POINT")
print(f"{'═'*60}")
print(f"  Requested : {LAT:.4f}°N, {LON:.4f}°E")
print(f"  Snapped to: {ACTUAL_LAT:.4f}°N, {ACTUAL_LON:.4f}°E")
print(f"  Pixel     : row={ROW}, col={COL}")
print(f"  Elevation : {ELEVATION:.1f} m")
print(f"  Slope     : {np.degrees(SLOPE):.2f}°")
print(f"  Aspect    : {np.degrees(ASPECT):.1f}°  (from north, clockwise)")
print(f"{'═'*60}")


In [ ]:
# ── Compute horizon profile (for topographic shadowing) ───────────────────────
from lunar.horizon import compute_horizon_profile, compute_sky_view_factor

N_AZ      = 360   # azimuth samples (360 = 1° resolution)
AZ_ANGLES = np.linspace(0, 2 * np.pi, N_AZ, endpoint=False, dtype=np.float32)

print("Computing horizon profile …")
HORIZONS = compute_horizon_profile(
    ROW, COL, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=3000
)
SVF = compute_sky_view_factor(HORIZONS)

print(f"✓ Horizon computed  ({N_AZ} directions)")
print(f"  Sky-view factor : {SVF:.3f}  (1.0 = unobstructed flat terrain)")
print(f"  Max horizon     : {np.degrees(HORIZONS.max()):.1f}°")


## 2b. Local Terrain Map

A global map of the Moon (left) and a zoomed view of the terrain around the
target location (right), coloured by elevation in kilometres.
The red star marks your target point and contour lines show terrain shape.

In [ ]:
fig = plots.dem_overview(ELEV_M, MAP_RES, ACTUAL_LAT, ACTUAL_LON, window_deg=5)
plt.show()

## 2c. Terrain Horizon Profile

A compass-rose (polar) view of how high the surrounding terrain appears from
the target pixel in every direction.

* **Flat terrain** → all values near 0 degrees.
* **Inside a crater** → high values in many directions.
* The **Sky-View Factor (SVF)** summarises how open the sky is
  (1.0 = completely unobstructed flat plain).

In [ ]:
fig = plots.horizon_polar(HORIZONS, AZ_ANGLES, SVF, ACTUAL_LAT, ACTUAL_LON)
plt.show()

## 2d. Illumination and Solar Flux Timeline

How much sunlight does this location receive over one full lunar day (29.5 Earth days)?

* **Panel 1** — Solar elevation: how high the Sun is above the horizon at each moment.
  Gold = sunlit; dark grey = in topographic shadow (sun is up but blocked by terrain);
  blue = nighttime.
* **Panel 2** — Absorbed solar energy (W/m2). Higher means the surface heats faster.
* **Panel 3** — Running fraction of the day that is actually sunlit.

> This cell evaluates solar geometry at 1000 time steps (takes a few seconds).

In [ ]:
fig = plots.illumination_timeline(
    ACTUAL_LAT, ACTUAL_LON,
    SLOPE, ASPECT,
    HORIZONS, AZ_ANGLES,
    sunscale=SUNSCALE, albedo=ALBEDO,
)
plt.show()

## 2e. Surface Temperature Map

An analytical estimate of surface temperatures across the local terrain.

* **Left** — Terrain elevation (for reference).
* **Centre** — Estimated peak (noon) temperature based on latitude and sunlight.
  The cyan star shows the full thermal-model result for the target point
  (after Section 3 has been run).
* **Right** — Estimated day-to-night temperature swing. The Moon swings
  up to ~300 K between day and night because there is no atmosphere to
  buffer the heat.

> The maps use a simplified formula; the full model (Section 3) is far more
> accurate for the single target point.

In [ ]:
# If Section 3 has already been run, annotate with the model peak temperature
T_max_surface = float(np.max(T_PROFILE[:, 0])) if 'T_PROFILE' in dir() else None

fig = plots.surface_temperature_map(
    ELEV_M, MAP_RES, ACTUAL_LAT, ACTUAL_LON,
    albedo=ALBEDO, emissivity=EMISSIVITY,
    window_deg=5,
    T_simulated_max=T_max_surface,
)
plt.show()

## 3. Run the Thermal Model

This runs the simulation. Expect ~15–30 s for 3 lunar days.

In [ ]:
import time

Z_GRID = solver.create_depth_grid()
print(f"Depth grid: {len(Z_GRID)} points, 0 – {Z_GRID[-1]:.1f} m")
print(f"Running {NDAYS} lunar day(s) …")

t0 = time.time()
T_PROFILE, T_ARR = solver.solve_thermal_model(
    z_grid     = Z_GRID,
    T_init     = T_INIT,
    lat_deg    = ACTUAL_LAT,
    lon_deg    = ACTUAL_LON,
    slope      = SLOPE,
    aspect     = ASPECT,
    horizons   = HORIZONS,
    az_angles  = AZ_ANGLES,
    chi        = CHI,
    model_id   = MODEL_ID,
    sunscale   = SUNSCALE,
    ndays      = NDAYS,
    albedo     = ALBEDO,
    emissivity = EMISSIVITY,
)
print(f"✓ Done in {time.time()-t0:.1f} s")
print(f"  Shape: {T_PROFILE.shape[0]} snapshots × {T_PROFILE.shape[1]} depth nodes")
print(f"  T range: {T_PROFILE.min():.1f} – {T_PROFILE.max():.1f} K")

# Extract statistics for the final lunar day
STATS = analysis.extract_stats(T_PROFILE, T_ARR, Z_GRID)
CYCLES = analysis.get_diurnal_cycles(T_PROFILE, T_ARR, Z_GRID,
                                      depths_m=[0.0, 0.1, 0.5, 1.0, 2.0])

print(f"\nFinal-day surface temperatures:")
print(f"  Min : {STATS['T_min'][0]:.1f} K")
print(f"  Max : {STATS['T_max'][0]:.1f} K")
print(f"  Mean: {STATS['T_mean'][0]:.1f} K")


## 4. Diurnal Temperature Cycles

Temperature vs time at several depths over the last simulated lunar day.
The surface (0 cm) shows the extreme swing; deeper layers barely vary.


In [ ]:
fig = plots.diurnal_cycles(
    cycles     = CYCLES,
    lat        = ACTUAL_LAT,
    lon        = ACTUAL_LON,
    model_name = MODEL,
    sunscale   = SUNSCALE,
)
plt.show()


## 5. Temperature Heatmap — Depth × Time

A 2-D view of how the diurnal heat wave penetrates into the subsurface.
Red = hot (day), dark = cold (night).  The penetration depth is ~0.5 m.


In [ ]:
fig = plots.heatmap(
    T_profile  = T_PROFILE,
    t_arr      = T_ARR,
    z_grid     = Z_GRID,
    lat        = ACTUAL_LAT,
    lon        = ACTUAL_LON,
    model_name = MODEL,
    depth_limit= 1.5,       # show top 1.5 m only
    colormap   = 'hot',
)
plt.show()


## 5b. Apollo HFE Data Analysis

This section loads the **actual Apollo Heat Flow Experiment (HFE) probe time-series** recorded over the operational lifetime of each mission.

### What the plots show

**Temperature history (left figure)**  
Each line is one sensor. The initial high temperatures immediately after emplacement reflect **residual frictional heat** from the rotary-percussive drilling. Temperatures decay exponentially toward equilibrium over several months. The green-shaded region marks the *stable window* (last 25 % of each probe's record) whose median is used as the validation reference — instead of a naive full-record average that would bias results high.

**Thermal shunting evidence (right figure)**  
- *Left panel*: T–depth profiles at successive time snapshots show how the disturbed early profile converges toward the true near-isothermal equilibrium.
- *Right panel*: The temperature difference ΔT between the upper (A) and lower (B) sensor of each gradient bridge traces the **thermal shunting** effect. The probe steel casing has conductivity ~100× higher than the surrounding regolith, so heat short-circuits along the probe body, suppressing the apparent gradient. The large initial ΔT collapses rapidly as drilling heat dissipates, then settles to a small residual that reflects the combined true geothermal gradient and the shunting offset.

In [ ]:
for _site in ('Apollo 15', 'Apollo 17'):
    print(f'\n{"─"*55}\n{_site}\n{"─"*55}')

    # ── Temperature time-series ───────────────────────────────────────
    fig_ts = plots.hfe_timeseries(_site)
    plt.show()

    # ── Thermal disturbance & shunting evidence ───────────────────────
    fig_sh = plots.hfe_shunting(_site)
    plt.show()

## 6. Apollo Site Validation

Runs the thermal model at **both** Apollo 15 (26.13°N, 3.63°E) and Apollo 17 (20.19°N, 30.77°E) and compares model predictions against the actual drill-hole temperature measurements from the Apollo Heat Flow Experiments.

**RMSE < 1 K** is an excellent fit.  If both sites run cold, increase `SUNSCALE` or decrease `ALBEDO` in Section 1 and re-run Section 3.


In [ ]:
# Section 6 always validates against BOTH Apollo sites, regardless of the
# current location.  Each site is solved at its own coordinates so the
# comparison is always scientifically meaningful.

from tqdm.auto import tqdm
from lunar import solver as _solver, dem as _dem, horizon as _hor

APOLLO_COORDS = {
    'Apollo 15': {'lat': 26.1323, 'lon':  3.6285},
    'Apollo 17': {'lat': 20.1911, 'lon': 30.7723},
}

APOLLO_RESULTS = {}   # populated below

for _site, _coords in APOLLO_COORDS.items():
    print(f"\n── {_site}  ({_coords['lat']}°N, {_coords['lon']}°E) ──")

    # Snap to DEM and get terrain geometry
    _row, _col, _alat, _alon, _elev, _sl, _asp = _dem.extract_point(
        _coords['lat'], _coords['lon'], ELEV_M, PIXEL_M, MAP_RES
    )

    # Horizon profile for this site
    from lunar.horizon import compute_horizon_profile
    _horiz = compute_horizon_profile(_row, _col, ELEV_M, PIXEL_M,
                                     AZ_ANGLES, max_range_px=1500)

    # Run the thermal model at the Apollo site coordinates
    _TP, _TA = solver.solve_thermal_model(
        Z_GRID, T_INIT,
        _alat, _alon, _sl, _asp, _horiz, AZ_ANGLES,
        CHI, MODEL_ID, SUNSCALE, NDAYS,
        albedo=ALBEDO, emissivity=EMISSIVITY,
    )

    _stats  = analysis.extract_stats(_TP, _TA, Z_GRID)
    _errors = analysis.compute_apollo_errors(_stats['T_mean'], Z_GRID, _site)

    APOLLO_RESULTS[_site] = {'stats': _stats, 'errors': _errors}

    print(f"   RMSE = {_errors['rmse']:.3f} K   Bias = {_errors['bias']:+.3f} K")

# ── Combined dual-site figure ────────────────────────────────────────────────
fig = plots.dual_apollo_comparison(
    apollo_results = APOLLO_RESULTS,
    model_name     = MODEL,
    sunscale       = SUNSCALE,
    chi            = CHI,
    albedo         = ALBEDO,
)
plt.show()


## 7. Model Comparison — Discrete vs Hayne 2017

Runs both density models with the same energy-balance parameters and
shows the differences in temperature profiles, amplitude, and (if at an
Apollo site) validation errors.


In [ ]:
# ── Configuration for comparison ─────────────────────────────────────────────
COMPARE_SUNSCALE   = SUNSCALE
COMPARE_ALBEDO     = ALBEDO
COMPARE_CHI        = CHI
COMPARE_EMISSIVITY = EMISSIVITY
COMPARE_NDAYS      = NDAYS

# Always compare at both Apollo sites so both sets of measurements appear
_COMPARE_SITES = {
    'Apollo 15': {'lat': 26.1323, 'lon':  3.6285},
    'Apollo 17': {'lat': 20.1911, 'lon': 30.7723},
}

for _site_name, _site_coords in _COMPARE_SITES.items():
    _lat, _lon = _site_coords['lat'], _site_coords['lon']
    print(f"\n{'─'*55}")
    print(f"Model comparison at {_site_name}  ({_lat}°N, {_lon}°E)")

    # DEM geometry for this site
    _row, _col, _alat, _alon, _elev, _sl, _asp = dem.extract_point(
        _lat, _lon, ELEV_M, PIXEL_M, MAP_RES
    )
    from lunar.horizon import compute_horizon_profile as _chp
    _horiz = _chp(_row, _col, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=1500)

    comparison_results = {}
    comparison_errors  = {}

    for model_key in ['discrete', 'hayne_exponential']:
        mid = models.MODEL_ID_MAP[model_key]
        print(f"  → {model_key} …", end=" ", flush=True)
        t0 = time.time()
        TP, TA = solver.solve_thermal_model(
            Z_GRID, T_INIT,
            _alat, _alon, _sl, _asp, _horiz, AZ_ANGLES,
            COMPARE_CHI, mid, COMPARE_SUNSCALE, COMPARE_NDAYS,
            albedo=COMPARE_ALBEDO, emissivity=COMPARE_EMISSIVITY,
        )
        st = analysis.extract_stats(TP, TA, Z_GRID)
        comparison_results[model_key] = st
        comparison_errors[model_key] = analysis.compute_apollo_errors(
            st['T_mean'], Z_GRID, _site_name
        )
        print(f"{time.time()-t0:.1f} s  "
              f"RMSE={comparison_errors[model_key]['rmse']:.2f} K  "
              f"Bias={comparison_errors[model_key]['bias']:+.2f} K")

    fig = plots.model_comparison(
        results_dict  = comparison_results,
        z_grid        = Z_GRID,
        lat           = _alat,
        lon           = _alon,
        apollo_errors = comparison_errors,
    )
    fig.suptitle(f'Model Comparison — Discrete vs Hayne 2017\n{_site_name}  '
                 f'({_alat:.3f}°N, {_alon:.3f}°E)',
                 fontsize=14, weight='bold', y=0.99)
    plt.show()


## 8. Parameter Sensitivity Analysis

Sweep one parameter across a range of values to see how much it affects
the temperature profile.  Good for finding the optimal value that minimises
RMSE vs Apollo measurements.

**Change `SENSITIVITY_PARAM`** (and optionally the range) then re-run.


In [ ]:
import numpy as np

# ── Choose parameter ──────────────────────────────────────────────────────────
SENSITIVITY_PARAM = 'sunscale'   # ← change this
# Options: 'sunscale' | 'albedo' | 'emissivity' | 'chi' | 'h_parameter' | 'rho_surface'

# ── Ranges ────────────────────────────────────────────────────────────────────
PARAM_RANGES = {
    'sunscale':    np.linspace(0.95, 1.20, 8),
    'albedo':      np.linspace(0.07, 0.14, 8),
    'emissivity':  np.linspace(0.90, 0.98, 8),
    'chi':         np.linspace(1.5,  4.0,  8),
    'h_parameter': np.linspace(0.03, 0.15, 8),   # metres (3–15 cm)
    'rho_surface': np.linspace(800,  1400,  8),  # kg/m³ (800–1400)
}

SENS_VALUES = PARAM_RANGES[SENSITIVITY_PARAM]

print(f"Sensitivity sweep: {SENSITIVITY_PARAM}")
print(f"Range: {SENS_VALUES[0]:.3g} → {SENS_VALUES[-1]:.3g}  ({len(SENS_VALUES)} values)")
print()

SENS_RESULTS = analysis.run_sensitivity(
    param_name   = SENSITIVITY_PARAM,
    param_values = SENS_VALUES,
    z_grid       = Z_GRID,
    T_init       = T_INIT,
    lat_deg      = ACTUAL_LAT,
    lon_deg      = ACTUAL_LON,
    slope        = SLOPE,
    aspect       = ASPECT,
    horizons     = HORIZONS,
    az_angles    = AZ_ANGLES,
    chi          = CHI,
    model_id     = MODEL_ID,
    sunscale     = SUNSCALE,
    ndays        = NDAYS,
    albedo       = ALBEDO,
    emissivity   = EMISSIVITY,
    baseline_h           = H_PARAM,
    baseline_rho_surface = 1100.0,
    apollo_site  = APOLLO_SITE if 'APOLLO_SITE' in dir() else None,
)

fig = plots.sensitivity_sweep(
    sens_results = SENS_RESULTS,
    param_name   = SENSITIVITY_PARAM,
    z_grid       = Z_GRID,
    lat          = ACTUAL_LAT,
    lon          = ACTUAL_LON,
    model_name   = MODEL,
)
plt.show()


## 9. Single Point Analysis

Run **two models** at one or more custom locations and compare their temperature
profiles.  If a location matches an Apollo site, validation errors are shown
in the same consistent style as Section 7.

**Add or remove entries in `LOCATIONS`** to change sites.


In [ ]:
# ── Sites to analyse ──────────────────────────────────────────────────────────
LOCATIONS = [
    {'name': 'Apollo 15',  'lat': 26.1323, 'lon':  3.6285},
    {'name': 'Apollo 17',  'lat': 20.1911, 'lon': 30.7723},
    # Add more: {'name': 'My Site', 'lat': 0.0, 'lon': 0.0},
]

# ── Model A ───────────────────────────────────────────────────────────────────
MODEL_A       = 'hayne_exponential'
A_SUNSCALE    = 1.0
A_ALBEDO      = 0.09
A_CHI         = 2.7
A_EMISSIVITY  = 0.95
A_NDAYS       = 3
A_H           = 0.07   # metres

# ── Model B ───────────────────────────────────────────────────────────────────
MODEL_B       = 'discrete'
B_SUNSCALE    = 1.10
B_ALBEDO      = 0.09
B_CHI         = 2.7
B_EMISSIVITY  = 0.95
B_NDAYS       = 3
B_H           = 0.07   # metres

# ── Run and plot for each location ────────────────────────────────────────────
for loc in LOCATIONS:
    name     = loc['name']
    lat_loc  = loc['lat']
    lon_loc  = loc['lon']

    print(f"\n{'─'*55}")
    print(f"Processing: {name}  ({lat_loc}°N, {lon_loc}°E)")

    # Extract DEM geometry
    row_l, col_l, alat, alon, elev_l, sl_l, asp_l = dem.extract_point(
        lat_loc, lon_loc, ELEV_M, PIXEL_M, MAP_RES
    )

    # Horizon
    horiz_l = compute_horizon_profile(
        row_l, col_l, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=1500
    )

    # Use requested coords (not DEM-snapped alat/alon) to avoid tolerance miss
    apollo_site_l = analysis.find_apollo_site(lat_loc, lon_loc)

    # Run Model A
    models.set_hayne_h(A_H); models.set_layer1_h(A_H)
    mid_A = models.MODEL_ID_MAP[MODEL_A]
    TP_A, TA_A = solver.solve_thermal_model(
        Z_GRID, T_INIT, alat, alon, sl_l, asp_l, horiz_l, AZ_ANGLES,
        A_CHI, mid_A, A_SUNSCALE, A_NDAYS,
        albedo=A_ALBEDO, emissivity=A_EMISSIVITY,
    )
    stats_A = analysis.extract_stats(TP_A, TA_A, Z_GRID)
    err_A   = (analysis.compute_apollo_errors(stats_A['T_mean'], Z_GRID, apollo_site_l)
               if apollo_site_l else None)

    # Run Model B
    models.set_hayne_h(B_H); models.set_layer1_h(B_H)
    mid_B = models.MODEL_ID_MAP[MODEL_B]
    TP_B, TA_B = solver.solve_thermal_model(
        Z_GRID, T_INIT, alat, alon, sl_l, asp_l, horiz_l, AZ_ANGLES,
        B_CHI, mid_B, B_SUNSCALE, B_NDAYS,
        albedo=B_ALBEDO, emissivity=B_EMISSIVITY,
    )
    stats_B = analysis.extract_stats(TP_B, TA_B, Z_GRID)
    err_B   = (analysis.compute_apollo_errors(stats_B['T_mean'], Z_GRID, apollo_site_l)
               if apollo_site_l else None)

    # Report
    if err_A:
        print(f"  Model A RMSE: {err_A['rmse']:.3f} K  bias: {err_A['bias']:+.3f} K")
    if err_B:
        print(f"  Model B RMSE: {err_B['rmse']:.3f} K  bias: {err_B['bias']:+.3f} K")

    # Plot — uses same function as Section 7
    res_dict = {MODEL_A: stats_A, MODEL_B: stats_B}
    err_dict = {MODEL_A: err_A, MODEL_B: err_B} if apollo_site_l else None

    fig = plots.model_comparison(
        results_dict  = res_dict,
        z_grid        = Z_GRID,
        lat           = alat,
        lon           = alon,
        apollo_errors = err_dict,
    )
    fig.suptitle(f'Single Point Analysis: {name}', fontsize=14, weight='bold', y=0.99)
    plt.show()

# Restore main model H-param after the loop
models.set_hayne_h(H_PARAM); models.set_layer1_h(H_PARAM)


## 10. Batch Processing

Process a list of locations automatically.  Results are collected in
`BATCH_RESULTS` (a list of dicts) and plotted with `plots.batch_summary()`.

**To load locations from a CSV file** (columns: name, lat, lon):
```python
import csv
with open('my_sites.csv') as f:
    BATCH_LOCATIONS = list(csv.DictReader(f))
```


In [ ]:
# ── Define batch locations ────────────────────────────────────────────────────
BATCH_LOCATIONS = [
    {'name': 'Apollo 15',    'lat': 26.1323, 'lon':  3.6285},
    {'name': 'Apollo 17',    'lat': 20.1911, 'lon': 30.7723},
    {'name': 'Equatorial 1', 'lat':  0.0,    'lon': 10.0   },
    {'name': 'Equatorial 2', 'lat':  0.0,    'lon': 45.0   },
    # Add more rows …
]

# ── Batch parameters ──────────────────────────────────────────────────────────
BATCH_SUNSCALE   = SUNSCALE
BATCH_ALBEDO     = ALBEDO
BATCH_EMISSIVITY = EMISSIVITY
BATCH_CHI        = CHI
BATCH_MODEL      = MODEL
BATCH_NDAYS      = NDAYS

print(f"Processing {len(BATCH_LOCATIONS)} location(s) …")
print(f"Model: {BATCH_MODEL}  SUNSCALE={BATCH_SUNSCALE}  ALBEDO={BATCH_ALBEDO}\n")

BATCH_RESULTS = analysis.run_batch(
    locations  = BATCH_LOCATIONS,
    elev_m     = ELEV_M,
    pixel_m    = PIXEL_M,
    map_res    = MAP_RES,
    z_grid     = Z_GRID,
    T_init     = T_INIT,
    chi        = BATCH_CHI,
    model_id   = models.MODEL_ID_MAP[BATCH_MODEL],
    sunscale   = BATCH_SUNSCALE,
    ndays      = BATCH_NDAYS,
    albedo     = BATCH_ALBEDO,
    emissivity = BATCH_EMISSIVITY,
)

# ── Summary table ─────────────────────────────────────────────────────────────
table = analysis.batch_to_table(BATCH_RESULTS)
print("\nResults summary:")
print(f"{'Name':<16} {'Lat':>7} {'Lon':>7} {'Elev':>7} {'T_min_0cm':>10} {'T_max_0cm':>10} {'T_mean_50cm':>12} {'RMSE':>7}")
print('─' * 85)
for i in range(len(table['name'])):
    rmse = table['RMSE_K'][i]
    rmse_str = f"{rmse:.3f}" if rmse is not None else "  —"
    print(f"{table['name'][i]:<16} {table['lat'][i]:>7.3f} {table['lon'][i]:>7.3f} "
          f"{table['elevation_m'][i]:>7.1f} "
          f"{table['T_min_0cm'][i]:>10.1f} {table['T_max_0cm'][i]:>10.1f} "
          f"{table['T_mean_50cm'][i]:>12.1f} {rmse_str:>7}")

# ── Batch plots ───────────────────────────────────────────────────────────────
fig = plots.batch_summary(BATCH_RESULTS, Z_GRID)
plt.show()


## 11. Density Model Viewer

Move the sliders to see how the regolith density changes with depth for
the current density model.  This updates instantly — no solver run needed.

* **Surface density** — The uppermost lunar soil is very fluffy (pulverised
  by micrometeorite impacts for billions of years). Apollo measurements
  suggest ~1100 kg/m3. Lower density means the surface temperature
  responds faster to sunlight because there is less mass to heat up.
* **Layer-1 thickness** — How deep the fluffy top layer extends before
  the soil becomes more compacted.

After choosing values here, copy them to Section 1b and re-run Section 3.

In [ ]:
import ipywidgets as widgets

@widgets.interact(
    rho_surface=widgets.IntSlider(
        value=1100, min=800, max=1400, step=50,
        description='Surface density (kg/m3)',
        style={'description_width': '210px'},
        layout=widgets.Layout(width='520px'),
    ),
    h_param=widgets.FloatSlider(
        value=H_PARAM, min=0.02, max=0.15, step=0.01,
        description='Layer thickness (m)',
        style={'description_width': '210px'},
        layout=widgets.Layout(width='520px'),
    ),
    model=widgets.Dropdown(
        options=['discrete', 'hayne_exponential'],
        value=MODEL,
        description='Density model',
        style={'description_width': '210px'},
    ),
)
def _density_viewer(rho_surface, h_param, model):
    fig = plots.density_profile(Z_GRID, model,
                                rho_surface=rho_surface,
                                h_param=h_param)
    plt.show()